# Qwen3.5-0.8B next-edit LoRA smoke on Kaggle T4 x2 (tinycomplete)
Pushed + monitored via the Kaggle API (`kaggle kernels push`). T4 has no bf16 path, so fp16 with GDN NaN watch.

In [ ]:
# Cell 1 — config
MODEL_ID = "Qwen/Qwen3.5-0.8B-Base"
MAX_SEQ_LENGTH = 2048
MAX_STEPS = 100
SEED = 42
PRECISION = "fp16"  # T4: no bf16; watch loss for NaN (GDN fp16 sensitivity)
DATASET_PATH = "/kaggle/input/tabcomplete-train/train_deepseek.jsonl"
OUT_DIR = "/kaggle/working/qwen35-lora-smoke"
print(MODEL_ID, MAX_SEQ_LENGTH, MAX_STEPS, SEED, PRECISION)

In [ ]:
# Cell 2 — runtime check
import torch
assert torch.cuda.is_available(), "GPU runtime required"
print(torch.__version__, torch.version.cuda)
print(torch.cuda.get_device_name(0))
assert torch.cuda.get_device_properties(0).total_memory / 2**30 > 10, "expected T4 x2-ish VRAM"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Cell 3 — repo + deps (keeps Kaggle CUDA torch)
!git clone --depth 1 https://github.com/Shlok-Bhakta/tabcomplete.git /kaggle/working/tabcomplete
!pip install -q unsloth "transformers>=5.2" pyyaml
import sys; sys.path.insert(0, "/kaggle/working/tabcomplete/src")
import unsloth, transformers, trl
print("unsloth", unsloth.__version__)
print("transformers", transformers.__version__)
print("trl", trl.__version__)

In [ ]:
# Cell 4 — dataset check (teacher labels attached as a Kaggle dataset)
from tinycomplete.train.train import load_jsonl_records, tokenize_records
from transformers import AutoTokenizer
from collections import Counter
recs = load_jsonl_records(DATASET_PATH)
assert len(recs) > 100, f"expected the full teacher set, got {len(recs)}"
tok = AutoTokenizer.from_pretrained(MODEL_ID)
ds, stats = tokenize_records(recs, tok, MAX_SEQ_LENGTH)
print("n/median/p95/max:", stats["n"], stats["median"], stats["p95"], stats["max"])
print("actions:", dict(Counter(r["action"] for r in recs)))
print("provenance:", dict(Counter(r["provenance"] for r in recs)))

In [ ]:
# Cell 5 — LoRA smoke
from tinycomplete.train.train import TrainConfig, train
cfg = TrainConfig(model_id=MODEL_ID, mode="lora", max_seq_length=MAX_SEQ_LENGTH,
                  max_steps=MAX_STEPS, seed=SEED, dataset_path=DATASET_PATH,
                  output_dir=OUT_DIR, precision=PRECISION)
report = train(cfg)
print(report)
assert report["tokens_per_second"] > 0
losses = report.get("extra", {})
print(losses)

In [ ]:
# Cell 6 — reload + before/after NLL on a fixed slice
import torch, torch.nn.functional as F
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from tinycomplete.train.train import load_jsonl_records, tokenize_records
tok = AutoTokenizer.from_pretrained(MODEL_ID)
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto")
ft = PeftModel.from_pretrained(base, OUT_DIR)
val, _ = tokenize_records(load_jsonl_records(DATASET_PATH)[-8:], tok, MAX_SEQ_LENGTH)
def nll(model, ids):
    t = torch.tensor([ids], device=model.device)
    with torch.no_grad():
        out = model(t)
    return F.cross_entropy(out.logits[0, :-1], t[0, 1:]).item()
for i, r in enumerate(val[:3]):
    print(f"ex{i}: base={nll(base, r['input_ids']):.3f} lora={nll(ft, r['input_ids']):.3f}")